# LLM expert persona eval analysis

## 1) The data

### Import results

In [1]:
import pandas as pd

df = pd.read_json('results.jsonl', lines=True)

df.head()

,result,category,is_expert,question,correct_answer,llm_answer,asked_at
0,parse_failure,math,True,Use divergence therem to evaluate $\iint_S \ve...,I,The Divergence Theorem states that $\iint_S \v...,2026-06-29 23:58:01+00:00
1,pass,law,True,"A buyer contracted in writing to purchase 1,00...",A,(A),2026-06-29 23:58:17+00:00
2,fail,business,True,"Suppose there are 8,000 hours in a year (actua...",G,(C),2026-06-29 23:58:32+00:00
3,pass,computer science,True,"Let a undirected graph G with edges E = {<0,1>...",H,The edges of the undirected graph G are given ...,2026-06-29 23:58:58+00:00
4,pass,math,True,What is the value of p in 24 = 2p?,H,(H),2026-06-29 23:59:14+00:00


### Number of rows

In [2]:
is_expert_len = len(df[df['is_expert']==True])
print(f'Total rows: {is_expert_len}')

Total rows: 50


### Parsing failure rate

In [3]:
(df['result'] == 'parse_failure').groupby(df['is_expert']).mean() * 100

is_expert
False    4.0
True     4.0
Name: result, dtype: float64

### Inspect parsing failures

In [4]:
df[df['result'] == 'parse_failure']['llm_answer']

0     The Divergence Theorem states that $\iint_S \v...
50    The passage states that Peter the Great was su...
79    The question is incomplete as the speed of sou...
99    The question is incomplete as the speed of sou...
Name: llm_answer, dtype: str

### Remove parsing failures

In [5]:
pre_filter_len = len(df)
df = df[df['result'] != 'parse_failure']
print(f'Rows removed: {pre_filter_len - len(df)}')

Rows removed: 4


## 2) Descriptive statistics

### Overall pass rate

In [6]:
(df['result'] == 'pass').groupby(df['is_expert']).mean() * 100

is_expert
False    50.000000
True     52.083333
Name: result, dtype: float64

### Pass rate by category

In [7]:
(df['result'] == 'pass').groupby([df['category'], df['is_expert']]).mean() * 100

category          is_expert
biology           False        100.000000
                  True         100.000000
business          False         42.857143
                  True          42.857143
chemistry         False          0.000000
                  True           0.000000
computer science  False         75.000000
                  True          75.000000
economics         False         25.000000
                  True          50.000000
engineering       False          0.000000
                  True          33.333333
health            False         25.000000
                  True          25.000000
history           True           0.000000
law               False         60.000000
                  True          60.000000
math              False         80.000000
                  True         100.000000
other             False         50.000000
                  True          50.000000
philosophy        False         33.333333
                  True           0.000000
physic

## 3) Chi-squared test

In [8]:
from scipy.stats import chi2_contingency

table = pd.crosstab(df['is_expert'], df['result'])

chi2, p, dof, expected = chi2_contingency(table)

print(table)
print('\np-value:', p)

result     fail  pass
is_expert            
False        24    24
True         23    25

p-value: 1.0
